# V14 — ViT-Small Backbone (tab:vit)
**Professor C**  
Goal: Run BMIA on ViT-Small/16 backbone fine-tuned on CIFAR-100.  
TTA updates **LayerNorm** (ViT equivalent of BatchNorm).  
Output → `V14_VIT.json`

**Kaggle**: Add Data → `hendrycks/cifar100c` | GPU T4 x2 | Internet ON | ~40 min

In [ ]:
import subprocess; subprocess.run(['pip','install','timm','--quiet'],check=True)

import os,json,copy,gc,time
import numpy as np
import torch,torch.nn as nn,torch.optim as optim
import torchvision,torchvision.transforms as T
from torch.utils.data import DataLoader
import timm

torch.manual_seed(42); np.random.seed(42)
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS=torch.cuda.device_count()
print(f'Device: {device}  GPUs: {N_GPUS}  timm: {timm.__version__}')

C100C_ROOT=None
for root,dirs,files in os.walk('/kaggle/input/'):
    if 'gaussian_noise.npy' in files and 'labels.npy' in files:
        C100C_ROOT=root; break
assert C100C_ROOT,'Add dataset: hendrycks/cifar100c'
print(f'CIFAR-100-C: {C100C_ROOT}')

NUM_CLASSES=100; IMG_SIZE=224
TTA_BATCH=32; EVAL_BATCH=128
LR=1e-5            # ViT requires small TTA LR
TAU=0.10; SEVERITY=5; SEV_OFFSET=(SEVERITY-1)*10000
CORRUPTIONS=['gaussian_noise','defocus_blur','snow','contrast','jpeg_compression']
METHODS=['tent','sar','eata','bmia_fixed','bmia_adaptive','bmia_adaptive_v3']

VIT_MEAN=torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
VIT_STD =torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)

# Loading raw uint8 + resize inline is slow — done per-corruption
def load_c100c(c):
    raw=np.load(f'{C100C_ROOT}/{c}.npy')[SEV_OFFSET:SEV_OFFSET+10000]
    lbl=np.load(f'{C100C_ROOT}/labels.npy')
    tf=T.Compose([T.ToPILImage(),T.Resize(224),T.ToTensor(),
                  T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
    X=torch.stack([tf(raw[i]) for i in range(len(raw))])
    return X.to(device),torch.tensor(lbl,dtype=torch.long).to(device)

print('Setup OK.')

In [ ]:
# ── Fine-tune ViT-Small/16 on CIFAR-100 ──────────────────────────
# Phase 1 (5ep): head only  |  Phase 2 (5ep): all params, low LR
model=timm.create_model('vit_small_patch16_224',pretrained=True,num_classes=NUM_CLASSES)
model=model.to(device)
print(f'ViT-S/16 params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')

ds=torchvision.datasets.CIFAR100('/tmp',train=True,download=True,transform=T.Compose([
    T.Resize(224),T.RandomHorizontalFlip(),T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])]))
dl=DataLoader(ds,batch_size=64,shuffle=True,num_workers=4,pin_memory=True)
crit=nn.CrossEntropyLoss()

# Phase 1: head only
for p in model.parameters(): p.requires_grad_(False)
for p in model.head.parameters(): p.requires_grad_(True)
opt1=optim.AdamW(model.head.parameters(),lr=1e-3,weight_decay=0.05)
model.train(); t0=time.time()
for ep in range(5):
    for x,y in dl:
        x,y=x.to(device),y.to(device)
        opt1.zero_grad(); crit(model(x),y).backward(); opt1.step()
    print(f'  Head ep={ep+1}/5  {(time.time()-t0)/60:.1f}min')

# Phase 2: all params
for p in model.parameters(): p.requires_grad_(True)
opt2=optim.AdamW(model.parameters(),lr=1e-4,weight_decay=0.05)
sch=optim.lr_scheduler.CosineAnnealingLR(opt2,T_max=5)
for ep in range(5):
    for x,y in dl:
        x,y=x.to(device),y.to(device)
        opt2.zero_grad(); crit(model(x),y).backward(); opt2.step()
    sch.step()
    print(f'  Full ep={ep+1}/5  {(time.time()-t0)/60:.1f}min')

model.eval(); SOURCE_STATE=copy.deepcopy(model.state_dict())
del opt1,opt2,dl,ds; gc.collect(); torch.cuda.empty_cache()
print(f'Done {(time.time()-t0)/60:.1f}min. SOURCE_STATE saved.')

In [ ]:
# ── Helpers (LayerNorm for ViT) ──────────────────────────────────
def freeze_except_ln(mdl):
    ids={id(p) for m in mdl.modules() if isinstance(m,nn.LayerNorm) for p in m.parameters()}
    for p in mdl.parameters(): p.requires_grad_(id(p) in ids)

def get_acc(mdl,X,y):
    mdl.eval(); preds=[]
    with torch.no_grad():
        for i in range(0,X.size(0),EVAL_BATCH):
            preds.append(mdl(X[i:i+EVAL_BATCH]).argmax(1))
    return round((torch.cat(preds)==y[:sum(p.size(0) for p in preds)]).float().mean().item(),4)

def run_method(X,y,method):
    model.load_state_dict(copy.deepcopy(SOURCE_STATE))
    model.train(); freeze_except_ln(model)
    params=[p for p in model.parameters() if p.requires_grad]
    opt=optim.AdamW(params,lr=LR,weight_decay=0.0)
    init={pn:p.data.clone() for pn,p in model.named_parameters() if p.requires_grad}
    lam=({'bmia_fixed':1.0,'bmia_adaptive':0.5,'bmia_adaptive_v3':0.1}).get(method,0.5)
    is_fixed=(method=='bmia_fixed'); use_v3=(method=='bmia_adaptive_v3')
    step=TTA_BATCH*2 if use_v3 else TTA_BATCH; fisher=None

    if method=='eata':
        fisher={pn:torch.zeros_like(p) for pn,p in model.named_parameters() if p.requires_grad}
        for j in range(0,min(64,X.size(0)),TTA_BATCH):
            xb=X[j:j+TTA_BATCH]
            if xb.size(0)<2: continue
            model.zero_grad(); pr=torch.softmax(model(xb),1)
            (-(pr*torch.log(pr+1e-8)).sum(1).mean()).backward()
            for pn,p in model.named_parameters():
                if pn in fisher and p.grad is not None: fisher[pn]+=p.grad.data**2*xb.size(0)
        for pn in fisher: fisher[pn]/=64
        model.load_state_dict(copy.deepcopy(SOURCE_STATE)); model.train(); freeze_except_ln(model)
        params=[p for p in model.parameters() if p.requires_grad]
        opt=optim.AdamW(params,lr=LR,weight_decay=0.0)

    for i in range(0,X.size(0),step):
        if use_v3:
            chunks=[X[i+s*TTA_BATCH:i+(s+1)*TTA_BATCH] for s in range(2)]
            chunks=[c for c in chunks if c.size(0)>=4]
            if not chunks: continue
            xb=torch.cat(chunks)
        else:
            xb=X[i:i+TTA_BATCH]
            if xb.size(0)<4: continue
        opt.zero_grad()
        pr=torch.softmax(model(xb),1); ent=-(pr*torch.log(pr+1e-8)).sum(1)
        if method=='tent': loss=ent.mean()
        elif method=='sar':
            mask=ent<0.4*np.log(NUM_CLASSES)
            if mask.sum()<2: continue
            loss=ent[mask].mean()
        elif method=='eata':
            mask=ent<0.4*np.log(NUM_CLASSES)
            if mask.sum()<2: continue
            fl=sum((fisher[pn]*(p-init[pn])**2).sum() for pn,p in model.named_parameters() if pn in fisher)
            loss=ent[mask].mean()+2000*fl
        else:
            mg=pr.mean(0); hy=-(mg*torch.log(mg+1e-8)).sum()
            anc=sum(((p-init[pn])**2).sum() for pn,p in model.named_parameters() if pn in init)
            loss=ent.mean()-hy+lam*anc
        loss.backward(); opt.step()
        if not is_fixed and method in('bmia_adaptive','bmia_adaptive_v3'):
            with torch.no_grad():
                dom=(torch.bincount(pr.argmax(1),minlength=NUM_CLASSES).float().max()/xb.size(0)).item()
            lam=min(10.,lam*1.1) if dom>TAU else max(0.01,lam*0.95)
    return get_acc(model,X,y)

print('LayerNorm TTA methods ready.')

In [ ]:
# ── Run ──────────────────────────────────────────────────────────
results=[]; t0=time.time()
for corr in CORRUPTIONS:
    print(f'\nLoading {corr} (resize 32→224 takes ~1min)...')
    X,y=load_c100c(corr)
    for m in METHODS:
        acc=run_method(X,y,m)
        results.append({'corruption':corr,'method':m,'acc':acc,'severity':SEVERITY})
        print(f'  {corr:<22} {m:<22} {acc*100:.1f}%')
        gc.collect(); torch.cuda.empty_cache()
    del X,y; gc.collect(); torch.cuda.empty_cache()
print(f'Done {(time.time()-t0)/60:.1f}min')

print('\nAvg per method:')
for m in METHODS:
    print(f'  {m:<22} {np.mean([r["acc"] for r in results if r["method"]==m])*100:.1f}%')

with open('V14_VIT.json','w') as f:
    json.dump({'experiment':'V14_VIT','backbone':'vit_small_patch16_224',
               'lr_tta':LR,'severity':SEVERITY,'corruptions':CORRUPTIONS,
               'results':results},f,indent=2)
print('Saved → V14_VIT.json')
with open('V14_VIT.json') as f: print(f.read())